# Comparing model quality with `evalstats`

**Scenario:** Imagine that 10 code review LLMs or agents each gave feedback on the same dataset of 200 pull requests and produced a detailed review. Every (model, item) pair has a `review_score` on a 1-5 Likert scale, which measures the quality of the PR code review for that item.

In [1]:
from pathlib import Path

import pandas as pd
from scipy.stats import pearsonr

import evalstats as es

CSV_PATH = Path("code_review_evalstats_demo.csv")

df = pd.read_csv(CSV_PATH)
evaldata = es.load_from(df)

df.head()

,model,item,review_score,response_length,expert_score
0,gemma-2-9b,item_1,3,140,NaN
1,gemma-2-9b,item_2,5,167,NaN
2,gemma-2-9b,item_3,3,180,NaN
3,gemma-2-9b,item_4,5,164,4.0
4,gemma-2-9b,item_5,3,157,NaN


## Step 1: Compare LLMs on code review quality

To compare models, we can pass the data to a `compare()` command. The args `p_values=True, omnibus=True` gets us the reporting a reviewer in ML or SWE might expect: a **Friedman** test (are ANY models different?) plus its standard post-hoc, **Wilcoxon signed-rank**, on every pairwise comparison of model performance.

In [2]:
result = es.compare(
    evaldata, factors="model", metric="review_score",
    p_values=True, omnibus=True,
)
result.summary()

Shape: BenchmarkShape(models=1, prompts=10, input_vars=1, evaluators=1)
Models: 10 | Inputs: 200

--- Robustness ---
                 mean  median       std        cv  iqr  cvar_10  p10  p25  p50  p75  p90
model                                                                                   
gemma-2-9b      3.710     4.0  0.705969  0.190288  1.0     2.75  3.0  3.0  4.0  4.0  5.0
llama-3.1-8b    3.475     3.0  0.763077  0.219590  1.0     2.20  3.0  3.0  3.0  4.0  4.0
llama-2-13b     3.000     3.0  0.729838  0.243279  0.0     1.90  2.0  3.0  3.0  3.0  4.0
gpt-oss-20b     3.120     3.0  0.669208  0.214490  1.0     1.95  2.0  3.0  3.0  4.0  4.0
yi-34b          4.020     4.0  0.641606  0.159603  0.0     2.95  3.0  4.0  4.0  4.0  5.0
qwen2.5-72b     3.355     3.0  0.693817  0.206801  1.0     2.15  3.0  3.0  3.0  4.0  4.0
claude-3-opus   3.710     4.0  0.698815  0.188360  1.0     2.65  3.0  3.0  4.0  4.0  5.0
gemini-1.5-pro  4.235     4.0  0.672336  0.158757  1.0     2.95  3.0  4.0  4.0  5.

# What do we learn? 

`gemini-1.5-pro` is the top model—not just by mean score, but significantly so, after all pairwise comparisons.

`evalstats` has calculated the "significant rank bands" and `gemini` is in a rank of its own, meaning its outperformance of all other models in the comparison is statistically significant at p<0.05.

### All of the CIs and p-values you see above were automatically FWER corrected. The choice of CIs and p-values methods here has been verified by simulations for this data type and sample size in LLM evals-like data scenarios. All you needed to do was pass `evalstats` the data. 

# We can conclude that `gemini` is the best, write up these results in our research paper, and go party!

![image](https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZ3o3cDU0cXhnaWMwb3k3MzUzNmY1amtjYzU2ZXR5eWg2cjJucmJxMSZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/lMVNl6XxTvXgs/giphy.gif)


.



.



.




.


.


---

...unfortunately, it's not the end of the story.

# Paper reviewers scrutinize your submission:

![image](https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExOTEya2RvOGEzMHFjOHRkc3Y1a2E3cDVjODAxMmNrbGZ2cDF5ajhzYiZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/10koGH0aisNoli/giphy.gif)

## Wait... where did `review_score` come from?

It wasn't human reviewers scoring all 2000 (model, item) pairs. 

## It's an **LLM judge**. 

#### "We used `gemini-1.5-pro` to judge the quality of all model outputs..."

"...and because we had 2000 outputs and recruiting humans is expensive and costly..."

Uh-oh!

![image](https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExMG16djd5amE2ZmszY2FvcXFnNGdhbnB1YnBjMGdwNXk4c3diaWt0YyZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/OlYEKypnEqv59zA1sQ/giphy.gif)

## Step 2 — Reviewers get mad at you, ask you for human validation

#### "Why did you use an LLM judge?"

#### "How do we trust these results??" 

## Step 3 — Validate the judge against the human gold set

You go out and get human experts to grade a randomly-chosen 30-item subset of each model's responses.

Now you can use `evalstats` to calculate some measure of alignment between the humans and LLM grades:

In [ ]:
alignment = es.judge_alignment(
    evaldata,
    llm_metric="review_score",
    human_groundtruth="expert_score",
)
alignment.summary()

# We need text in evalstats outputs that justifies explanation of why certain stats tests were used.

NOTE: Weighted Cohen's k is chosen here because we're dealing with Likert data, which still gives credit for 'close' ratings.

### You report this alignment in your paper, arguing that "0.51 indicates **moderate agreement**" and hence it's a relatively aligned judge—not great, but enough to trust.

# Are you done??

Most people in HCI and SWE in published papers: 

![image](https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZ2xoNHI5ZWZyYnNmdHA4YjBkM3J6bnZudmppZGt6czJneGI3aXVxdCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/tZpGRRMUoXgeQ/giphy.gif)

Typically, people stop there. 

# But then why should we trust your p-values and statistical results? Isn't it still just nonsense?

## Step 4 — Re-run the SAME comparison, corrected for judge bias

Can we correct for judge bias?

`evalstats` includes corrections, not just to CIs and point estimates, but for common statistical tests like Friedman and Wilcoxon signed-ranks. You simply do the same `compare()` call as before and just pass `alignment=` to the metric. 

This uses Prediction-Powered Inference (PPI) to combine the biased LLM
judge scores with the unbiased-but-scarce human labels. The Friedman test
and the Wilcoxon signed ranks p-values are both PPI-corrected too, not just
the means and CIs.

In [4]:
result_corrected = es.compare(
    evaldata,
    factors="model",
    metric="review_score",
    p_values=True, omnibus=True,
    alignment={"review_score": alignment},
)
result_corrected.summary()

Shape: BenchmarkShape(models=1, prompts=10, input_vars=1, evaluators=1)
Models: 10 | Inputs: 200

--- Robustness ---
                    mean  median       std        cv  iqr  cvar_10  p10  p25  p50  p75  p90
model                                                                                      
gemma-2-9b      2.476667     4.0  0.705969  0.190288  1.0     2.75  3.0  3.0  4.0  4.0  5.0
llama-3.1-8b    2.975000     3.0  0.763077  0.219590  1.0     2.20  3.0  3.0  3.0  4.0  4.0
llama-2-13b     3.166667     3.0  0.729838  0.243279  0.0     1.90  2.0  3.0  3.0  3.0  4.0
gpt-oss-20b     3.053333     3.0  0.669208  0.214490  1.0     1.95  2.0  3.0  3.0  4.0  4.0
yi-34b          3.153333     4.0  0.641606  0.159603  0.0     2.95  3.0  4.0  4.0  4.0  5.0
qwen2.5-72b     3.321667     3.0  0.693817  0.206801  1.0     2.15  3.0  3.0  3.0  4.0  4.0
claude-3-opus   3.743333     4.0  0.698815  0.188360  1.0     2.65  3.0  3.0  4.0  4.0  5.0
gemini-1.5-pro  3.635000     4.0  0.672336  0.158757  1

## Did the ranking actually change? What models did the judge get most wrong?

A side-by-side of judge-only vs. PPI-corrected means, sorted by the
corrected mean.

In [11]:
entities = result_corrected.to_dict()["variance_components"]["entities"]
rows = sorted(entities.items(), key=lambda kv: kv[1]["rectifier"])

print(f"{'Model':<16} {'Judge-only mean':>16} {'Corrected mean':>16} {'Rectifier':>12}")
for name, e in rows:
    print(
        f"{name:<16} {e['llm_mean']:>16.3f} {e['ppi_mean']:>16.3f} "
        f"{e['rectifier']:>+12.3f}"
    )

Model             Judge-only mean   Corrected mean    Rectifier
gemma-2-9b                  3.710            2.477       -1.233
yi-34b                      4.020            3.153       -0.867
gemini-1.5-pro              4.235            3.635       -0.600
llama-3.1-8b                3.475            2.975       -0.500
gpt-oss-20b                 3.120            3.053       -0.067
llama-2-70b                 3.995            3.962       -0.033
qwen2.5-72b                 3.355            3.322       -0.033
claude-3-opus               3.710            3.743       +0.033
llama-2-13b                 3.000            3.167       +0.167
falcon-40b                  3.940            4.140       +0.200


# How can we trust the PPI-corrected stats tests?

![image](../simulations/out/official_20260707_010008/plots/pvalues_ppi_reps300_20260707_010010_typeI_corrected_vs_uncorrected.png)

# Questions

### What kinds of statistics need to be run for, or around, AI evaluation results?
* What is missing here that you’d like to see? 
* Is there something that should be changed?
* What needs aren’t being met yet?
### What are the norms in SWE for reporting stats around AI evaluations (or what _should_ they be)? 
### How to “evaluate” this toolkit for a research paper (beyond running simulations on individual method choices)? 
	* For an HCI venue?
	* For a SWE venue? 
